# Bank´s request Practice

### Creating a PySpark DataFrame, schema inference, printSchema(), show(), and row counting.

### Objective: Read the CSV using PySpark by defining an explicit schema (without using `inferSchema=True`) and validate the number of rows and columns.

## Execute Spark Local Session

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## Import all libraries needed

In [19]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, regexp_replace, to_date, coalesce, lit, date_format, countDistinct
from pyspark.sql.types import StringType, StructType, StructField, DoubleType, IntegerType, DateType, IntegerType


## Create Spark Session

In [20]:
def create_spark_session(app_name: str = "DataFrame") -> SparkSession:
    spark = (
        SparkSession.builder
        .appName(app_name)
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.driver.memory", "2g")
        .master("local[*]")
        .getOrCreate()
    )
    
    return spark

## Create DataFrame's Schema and DataFrame from CSV file

In [22]:
def create_dataframe(spark: SparkSession, file_path: str) -> DataFrame:
    '''
    Create a DataFrame from a CSV file with a predefined schema below
    '''
    schema = StructType([
        StructField("transaction_id", StringType(), True),
        StructField("customer_id", StringType(), True),
        StructField("transaction_date", StringType(), True),
        StructField("transaction_type", StringType(), True),
        StructField("amount", DoubleType(), True),
        StructField("account_number", StringType(), True),
        StructField("branch_name", StringType(), True),
        StructField("employee_id", StringType(), True),
        StructField("customer_name", StringType(), True),
        StructField("email", StringType(), True),
        StructField("phone", IntegerType(), True),
        StructField("account_status", StringType(), True),
        StructField("risk_level", StringType(), True),
        StructField("product_type", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("country", StringType(), True),
        StructField("state", StringType(), True),
        StructField("balance", DoubleType(), True),
        StructField("credit_score", IntegerType(), True),
        StructField("last_login", IntegerType(), True)
    ])
    
    # Create a DataFrame from the CSV file with the specified schema
    df = (
        spark.read
        .csv(file_path, 
             header = True, 
             schema = schema,
             sep = ",")
    )
    
    return df


## Count all rows and columns we have from our dataframe

In [32]:
def columns_rows_metrics(df: DataFrame) ->DataFrame:
    '''
    Function to get the number of columns and rows in a DataFrame
    '''
    metrics_df = (
        df.groupBy().agg(count("*")
        .alias("num_rows"), 
        lit(len(df.columns)).alias("num_columns"))
    )
    
    return metrics_df

## Main function

In [33]:
def main():
    spark = create_spark_session()
    
    try:
        print(" ---- Original DataFrame ---- ")
        df = create_dataframe(spark, "C:/temp/dataset_bank.csv")
        df.show(5, truncate = False)
        print(" ---- Metrics ----")
        df_metrics = columns_rows_metrics(df)
        df_metrics.show(truncate = False)
    except Exception as Error:
        print(f"Error: {Error}")
        spark.stop()

if __name__ == "__main__":
    main()
    

 ---- Original DataFrame ---- 
+--------------+-----------+----------------+----------------+--------+--------------+-----------+-----------+-------------------+---------------------+-----+--------------+----------+------------+----------+-------------+-----+--------+------------+----------+
|transaction_id|customer_id|transaction_date|transaction_type|amount  |account_number|branch_name|employee_id|customer_name      |email                |phone|account_status|risk_level|product_type|channel   |country      |state|balance |credit_score|last_login|
+--------------+-----------+----------------+----------------+--------+--------------+-----------+-----------+-------------------+---------------------+-----+--------------+----------+------------+----------+-------------+-----+--------+------------+----------+
|TXN00000368   |CUST00328  |NULL            |Transfer        |NULL    |ACC0000000104 |SAN DIEGO  |NULL       |JENNIFER GARCIA    |NULL                 |NULL |Suspended     |low       